In [1]:
!pip install dash

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 45.5 MB/s eta 0:00:00


In [2]:
import dash
import json
import threading
from dash import dcc
from dash import html
from dash.dependencies import Input, Output
from google.colab import output, drive

In [5]:
app = dash.Dash()

app.layout = html.Div(
    children=[
              html.Button('My Button', id='button'),
              dcc.Slider(id='slider', max=10, min=0, step=2, value=4),
              dcc.Input(id='input', type="text", placeholder="type something here"),
              html.Div(id='result')
              ],
    style={
        "display": "inline-flex",
        "flex-direction": "column",
        "justify-content": "center",
        "align-content": "center",
        "height": "100%",
        "fontFamily": "verdana",
        "color": "#444"
        }
)

@app.callback(
    Output('result', 'children'),
    Input('button', 'n_clicks'),
    Input('slider', 'value'),
    Input('input', 'value'),
)
def show_context(clicks, slider_value, input_value):
    context = dash.callback_context  # informace, co callback spustilo

    # 1) Zatím se nic nestalo
    if not context.triggered:
        return "Ještě neproběhla žádná interakce – klikni na tlačítko, posuň slider nebo něco napiš."

    # 2) Zjistíme, který vstup callback spustil
    triggered = context.triggered[0]          # první (a jediný) záznam
    prop_id = triggered["prop_id"]           # např. "button.n_clicks"
    triggered_id, triggered_prop = prop_id.split(".")
    triggered_value = triggered["value"]

    # 3) Věta popisující, CO callback spustilo
    sentence_trigger = (
        f"Callback byl spuštěn prvkem '{triggered_id}', "
        f"konkrétně změnou vlastnosti '{triggered_prop}' na hodnotu {triggered_value}."
    )

    # 4) Věta popisující AKTUÁLNÍ hodnoty všech vstupů
    sentence_state = (
        f"Aktuální stav vstupů je:\n"
        f"- tlačítko: n_clicks = {clicks},\n"
        f"- slider: value = {slider_value},\n"
        f"- textový input: value = '{input_value}'."
    )

    # 5) Vrátíme hezky zformátovaný text
    children = [
        html.H4("Shrnutí callback contextu:"),
        html.P(sentence_trigger),
        html.Br(),
        html.P(sentence_state.replace("\n", " ")),  # v jedné větě
    ]

    return children


In [7]:
PORT = 8050  # klidně změň, když bude obsazený

def run_app():
    app.run(
        host="0.0.0.0",
        port=PORT,
        debug=False,
        use_reloader=False,
    )

thread = threading.Thread(target=run_app)
thread.start()

# zobrazí aplikaci přímo v buňce jako iframe
output.serve_kernel_port_as_iframe(port=PORT, height=400)

print("\n--- Souhrn ---")
print(f"1) Dash aplikace běží na portu {PORT}.")
print("2) Nad tímto textem vidíš vložené okno (iframe) s aplikací.")
print("3) Pokud iframe nevidíš, spusť buňku znovu nebo obnov stránku.")

Dash is running on http://0.0.0.0:8050/



INFO:dash.dash:Dash is running on http://0.0.0.0:8050/



<IPython.core.display.Javascript object>


--- Souhrn ---
1) Dash aplikace běží na portu 8050.
2) Nad tímto textem vidíš vložené okno (iframe) s aplikací.
3) Pokud iframe nevidíš, spusť buňku znovu nebo obnov stránku.


In [9]:
# ============================================
# Kontext zpětného volání – finální řešení
# ============================================

import dash
import threading
from dash import dcc, html
from dash.dependencies import Input, Output
from google.colab import output

# -----------------------
# 1) Inicializace aplikace
# -----------------------
app = dash.Dash()

# -----------------------
# 2) Layout aplikace
# -----------------------
app.layout = html.Div(
    children=[
        html.Button("My Button", id="button"),
        dcc.Slider(id="slider", max=100, min=0, step=2, value=40),
        dcc.Input(id="input", type="text", placeholder="type something here"),
        html.Div(id="result"),
    ],
    style={
        "display": "inline-flex",
        "flex-direction": "column",
        "justify-content": "center",
        "align-content": "center",
        "height": "100vh",
        "fontFamily": "verdana",
        "color": "#444",
    }
)

# -----------------------
# 3) Callback – práce s dash.callback_context
#     místo surových slovníků vrací věty
# -----------------------
@app.callback(
    Output("result", "children"),
    Input("button", "n_clicks"),
    Input("slider", "value"),
    Input("input", "value"),
)
def show_context(clicks, slider_value, input_value):
    context = dash.callback_context

    # 3.1) Zatím neproběhla žádná interakce
    if not context.triggered:
        return html.P(
            "Ještě neproběhla žádná interakce – klikni na tlačítko, "
            "posuň slider nebo něco napiš do textového pole."
        )

    # 3.2) Co callback spustilo
    triggered = context.triggered[0]        # první (jediný) záznam
    prop_id = triggered["prop_id"]          # např. "button.n_clicks"
    triggered_id, triggered_prop = prop_id.split(".")
    triggered_value = triggered["value"]

    sentence_trigger = (
        f"Callback byl spuštěn prvkem „{triggered_id}“, "
        f"protože se změnila jeho vlastnost „{triggered_prop}“ "
        f"na hodnotu {triggered_value}."
    )

    # 3.3) Jaké jsou aktuální hodnoty všech vstupů
    # ošetření None, aby text vypadal hezky
    clicks_txt = clicks if clicks is not None else 0
    slider_txt = slider_value if slider_value is not None else "nezadáno"
    input_txt = input_value if input_value not in (None, "") else "prázdný text"

    sentence_state = (
        "Aktuální stav vstupů je následující: "
        f"tlačítko má n_clicks = {clicks_txt}, "
        f"slider má hodnotu {slider_txt} "
        f"a textový input obsahuje „{input_txt}“."
    )

    return [
        html.H4("Shrnutí callback contextu:"),
        html.P(sentence_trigger),
        html.P(sentence_state),
    ]

# -----------------------
# 4) Spuštění aplikace v Colabu + iframe + localhost odkaz
# -----------------------
PORT = 8050  # pokud by byl port obsazený, můžeš změnit např. na 8060

def run_app():
    app.run(
        host="0.0.0.0",
        port=PORT,
        debug=False,
        use_reloader=False,
    )

# spuštění Dash aplikace v separátním vlákně
thread = threading.Thread(target=run_app)
thread.start()

# vložení aplikace do výstupu buňky jako iframe
output.serve_kernel_port_as_iframe(port=PORT, height=450)

print("\n--- Souhrn ---")
print(f"1) Dash aplikace běží na portu {PORT}.")
print(f"2) Odkaz localhost (mimo Colab prostředí): http://127.0.0.1:{PORT}/")
print("3) V tomto notebooku vidíš aplikaci přímo nad tímto textem v iframe.")
print("4) Pokud iframe nevidíš, spusť buňku znovu nebo obnov stránku.")

Dash is running on http://0.0.0.0:8050/



INFO:dash.dash:Dash is running on http://0.0.0.0:8050/



 * Serving Flask app '__main__'


<IPython.core.display.Javascript object>


--- Souhrn ---
1) Dash aplikace běží na portu 8050.
2) Odkaz localhost (mimo Colab prostředí): http://127.0.0.1:8050/
3) V tomto notebooku vidíš aplikaci přímo nad tímto textem v iframe.
4) Pokud iframe nevidíš, spusť buňku znovu nebo obnov stránku.
 * Debug mode: off


Address already in use
Port 8050 is in use by another program. Either identify and stop that program, or start the server with a different port.


Změňte způsob fungování aplikace tak, aby zobrazovala rozbalovací seznam místo tlačítka, posuvníku a textového pole. Jak se změní obsah každého slovníku?


In [10]:
# ============================================
# Kontext zpětného volání – cvičení 2
# ============================================

# Nová aplikace pro druhé cvičení
app2 = dash.Dash()

# Layout: jen dropdown + div na výpis výsledků
app2.layout = html.Div(
    children=[
        html.H3("Context callback – cvičení 2"),
        html.P("Vyber možnost z rozbalovacího seznamu:"),
        dcc.Dropdown(
            id="dropdown",
            options=[
                {"label": "Možnost A", "value": "A"},
                {"label": "Možnost B", "value": "B"},
                {"label": "Možnost C", "value": "C"},
            ],
            value="A",
            clearable=False,
            style={"width": "300px"}
        ),
        html.Br(),
        html.Div(id="result2"),
    ],
    style={
        "display": "inline-flex",
        "flex-direction": "column",
        "justify-content": "center",
        "align-content": "center",
        "height": "100vh",
        "fontFamily": "verdana",
        "color": "#444",
    }
)


In [11]:
@app2.callback(
    Output("result2", "children"),
    Input("dropdown", "value"),
)
def show_context_dropdown(selected_value):
    ctx = dash.callback_context

    # Ještě žádná interakce
    if not ctx.triggered:
        return html.P("Zatím žádná interakce – změň hodnotu v rozbalovacím seznamu.")

    # Slovníky z callback contextu
    triggered = ctx.triggered
    inputs = ctx.inputs
    states = ctx.states  # u tohoto příkladu bude prázdný

    children = [
        html.H4("1) triggered – co callback spustilo:"),
        html.Pre(json.dumps(triggered, indent=2, ensure_ascii=False)),

        html.H4("2) inputs – aktuální hodnoty vstupů:"),
        html.Pre(json.dumps(inputs, indent=2, ensure_ascii=False)),

        html.H4("3) states – stavové hodnoty (v tomto příkladu prázdné):"),
        html.Pre(json.dumps(states, indent=2, ensure_ascii=False)),

        html.Br(),
        html.P(f"Z rozbalovacího seznamu je aktuálně vybráno: {selected_value}"),
    ]

    return children


In [12]:
# -----------------------
# Spuštění app2 v Colabu
# -----------------------
PORT2 = 8060  # jiný port než u cvičení 1

def run_app2():
    app2.run(
        host="0.0.0.0",
        port=PORT2,
        debug=False,
        use_reloader=False,
    )

thread2 = threading.Thread(target=run_app2)
thread2.start()

# vložení aplikace do výstupu buňky jako iframe
output.serve_kernel_port_as_iframe(port=PORT2, height=450)

print("\n--- Souhrn – cvičení 2 ---")
print(f"1) Druhá Dash aplikace běží na portu {PORT2}.")
print(f"2) Odkaz localhost (mimo Colab prostředí): http://127.0.0.1:{PORT2}/")
print("3) V tomto notebooku vidíš aplikaci výše v iframe.")
print("4) Pokud iframe nevidíš, spusť buňku znovu nebo obnov stránku.")


Dash is running on http://0.0.0.0:8060/



INFO:dash.dash:Dash is running on http://0.0.0.0:8060/



<IPython.core.display.Javascript object>


--- Souhrn – cvičení 2 ---
1) Druhá Dash aplikace běží na portu 8060.
2) Odkaz localhost (mimo Colab prostředí): http://127.0.0.1:8060/
3) V tomto notebooku vidíš aplikaci výše v iframe.
4) Pokud iframe nevidíš, spusť buňku znovu nebo obnov stránku.
 * Serving Flask app '__main__'
 * Debug mode: off


🔵 1) Cv. 1

Aplikace měla 3 vstupy:
✔ tlačítko (button.n_clicks)
✔ slider (slider.value)
✔ text input (input.value)

Callback vždy vracel tři slovníky:

triggered → co spustilo callback

inputs → aktuální hodnoty všech vstupů

states → seznam „stavových“ hodnot (v tomto cvičení prázdný)

triggered se měnil podle toho, na co uživatel klikl / co změnil.

-> cílem bylo pochopit jak Dash rozhoduje, co callback spustilo.

Klíčová informace:
Slovníky jsou plné dat, protože máme víc než jeden vstup.

🔵 2) Cv 2

Aplikace má teď jen jeden vstup:
✔ Dropdown

Callback opět vrací triggered, inputs, states, ale:

triggered bude vždy mít jen jeden prvek → dropdown.value

inputs bude mít jen jeden klíč

states je opět prázdný

Při změně volby uvidíš krásně jednodušší strukturu callback contextu.

Klíčová změna:
Protože je jen jeden vstup, slovníky jsou extrémně jednoduché → dělá to zřetelný rozdíl oproti cvičení 1.


callback_context.triggered říká jen a pouze, co naposledy spustilo callback.

Je zásadní při stavbě komplikovanějších callbacků.

Pokud aplikace používá jediný vstup → celý context je velmi jednoduchý.

Pokud má více vstupů → context ukazuje, který z nich callback aktivoval.